In [1]:
# conda activate anndata

import re
import sys
import pickle
import numpy as np
import pandas as pd

sys.path.append("code")
sys.path.append("/mnt/lareaulab/reliscu/code")

from process_gtf import *
from empirical_corr_pvals import *
from junction2psi import *

In [2]:
data_source = "GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed"
psi = pd.read_csv(f"data/GTEx_frontal_cortex_SE_PSI.csv", index_col=0)
eigengene_df = pd.read_csv("data/GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed_ct_eigengenes.csv", index_col=0)

##  Correlate PSI and eigengenes and calc empirical p-values

In [3]:
# Make sure order of samples matches
eigengene_df.index = eigengene_df.index.str.replace(".", "-") 
common = psi.columns.intersection(eigengene_df.index)
psi = psi[common]
eigengene_df = eigengene_df.loc[common]

In [4]:
# --- correlations + permutation p-values + FDR + CIs ---
(psi_corr_df, psi_pval_df, psi_fdr_df, psi_ci_lower_df, 
 psi_ci_upper_df, perm_corr_results) = spearman_permutation_test(
    psi,
    eigengene_df,
    n_perms=10000,
    ci=0.95
)

Done: CGE Class
Done: All GABAergic
Done: Micro/PVM
Done: Oligo
Done: Astro
Done: Endo
Done: All Neuronal
Done: Deep layer glutamatergic
Done: Upper layer glutamatergic
Done: OPC


In [5]:
results = {
    "psi_corr_df": psi_corr_df,
    "psi_pval_df": psi_pval_df,
    "psi_fdr_df": psi_fdr_df,
    "psi_ci_lower_df": psi_ci_lower_df,
    "psi_ci_upper_df": psi_ci_upper_df,
    "perm_corr_results": perm_corr_results,
}

with open(f"data/{data_source}_spearman_permutation_test.pkl", "wb") as f:
    pickle.dump(results, f)

In [ ]:
steiger_results = compare_all_ct_pairs(
    psi_corr_df,
    perm_corr_results,
    eigengene_df.columns.tolist()
)

In [ ]:
# For correlation difference test:
ct_hierarchy = {
    # broader  cell classes should not be comapred against their subtypes (subtypes listed as children);
    'All Neuronal':              ['All GABAergic', 'CGE Class','Upper layer glutamatergic', 'Deep layer glutamatergic'],
    'All GABAergic':             ['CGE Class'],
    # subtypes must be significantly greater than everything
    'CGE Class':                 [],
    'Upper layer glutamatergic': [],
    'Deep layer glutamatergic':  [],
    'Oligo':                     [],
    'OPC':                       [],
    'Astro':                     [],
    'Micro/PVM':                 [],
    'VLMC':                      [],
    'Endo':                      [],
    'Peri':                      [],
}

In [ ]:
print("find_specific_SEs_per_ct_basic:")
print("")

ct_specific_SEs_basic = find_specific_SEs_per_ct_basic(
    psi_fdr_df,
    psi_corr_df,
    fdr_thresh=0.05
)

print("")
print("find_specific_SEs_per_ct:")

ct_specific_SEs_strict = {}

for ascending in (True, False):
    print("")
    print(ascending)
    ct_specific_SEs_strict[str(ascending)] = find_specific_SEs_per_ct(
        steiger_results,
        psi_fdr_df,
        psi_corr_df,
        ct_hierarchy,
        fdr_thresh=0.05,
        ascending=ascending
    )

find_specific_SEs_per_ct_basic:

CGE Class: 5773 significant SEs
All GABAergic: 7536 significant SEs
Micro/PVM: 62 significant SEs
Oligo: 579 significant SEs
Astro: 6223 significant SEs
Endo: 2851 significant SEs
All Neuronal: 8566 significant SEs
Deep layer glutamatergic: 5580 significant SEs
Upper layer glutamatergic: 6341 significant SEs
OPC: 362 significant SEs

find_specific_SEs_per_ct:

True
CGE Class: 0 specific SEs (5773 significant total)
All GABAergic: 4 specific SEs (7536 significant total)
Micro/PVM: 5 specific SEs (62 significant total)
Oligo: 17 specific SEs (579 significant total)
Astro: 1005 specific SEs (6223 significant total)
Endo: 3 specific SEs (2851 significant total)
All Neuronal: 1593 specific SEs (8566 significant total)
Deep layer glutamatergic: 2 specific SEs (5580 significant total)
Upper layer glutamatergic: 3 specific SEs (6341 significant total)
OPC: 2 specific SEs (362 significant total)

False
CGE Class: 0 specific SEs (5773 significant total)
All GABAe

In [ ]:
# --- combine results ---
combined_results = combine_results(
    ct_specific_SEs_basic,
    ct_specific_SEs_strict,
    psi_corr_df,
    psi_fdr_df,
    steiger_results,
    ct_hierarchy
)

## Annotate and save results per cell type

In [ ]:
# # Parse GTF

# exclude = ""
# gene_type = "all"
# no_trim_id = False
# gene_type_tag = "gene_type"
# transcript_type_tag = "transcript_type"

# gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v50.annotation.gtf"
# gtf = process_gtf(gtf_file, exclude, no_trim_id, gene_type_tag, transcript_type_tag)
# gtf_gene = gtf[gtf.feature == "gene"]

# pickle.dump(gtf, open("data/gencode.v50.annotation_gtf_parsed.pkl", "wb"))

Processing GTF file...


INFO:root:Extracted GTF attributes: ['gene_id', 'gene_type', 'gene_name', 'level', 'tag', 'transcript_id', 'transcript_type', 'transcript_name', 'exon_number', 'exon_id', 'transcript_support_level', 'havana_transcript', 'hgnc_id', 'havana_gene', 'ont', 'protein_id', 'ccdsid', 'artif_dupl']


In [ ]:
gtf = pickle.load(open("data/gencode.v50.annotation_gtf_parsed.pkl", "rb"))
gtf_gene = gtf[gtf.feature == "gene"]

In [ ]:
# # Get genome coordinates for each SE

# intron_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/psix_annotation/intron_file_v50.tab.gz"
# intron_table = read_intron_file(intron_file)

# intron_coords_df = intron_table['intron'].str.split(r"[:\-]", expand=True).iloc[:, :4]
# intron_coords_df.columns = ["chr", "intron_first_base", "intron_last_base", "strand"]
# intron_coords_df.index = intron_table.index
# intron_coords_df['SE'] = intron_coords_df.index.str.split("_").str[:3].str.join("_")
# intron_coords_df = intron_coords_df[intron_coords_df['SE'].isin(psi.index)] # Subset to SEs in PSI data
# intron_coords_df['intron_first_base'] = intron_coords_df["intron_first_base"].astype(int)
# intron_coords_df['intron_last_base'] = intron_coords_df["intron_last_base"].astype(int)

# def safe_SE_coords(g):
#     i1 = g.loc[g.index.str.contains("I1$"), "intron_last_base"].values
#     i2 = g.loc[g.index.str.contains("I2$"), "intron_first_base"].values
#     if len(i1) == 0 or len(i2) == 0:
#         return pd.Series({"chr": None, "SE_start": None, "SE_end": None})
#     return pd.Series({
#         "chr": g["chr"].iloc[0],
#         "intron_end": i1[0],
#         "exon_start": i1[0] + 1,
#         "exon_end": i2[0] - 1,
#         "intron_start": i2[0],
#         "exon_len": i2[0] - i1[0] - 1
#     })

# SE_coords_df = intron_coords_df.groupby("SE").apply(safe_SE_coords)
# SE_coords_df['gene_id'] = SE_coords_df.index.str.split("_").str[0]
# SE_coords_df['exon_coords'] = SE_coords_df['chr'] + ":" \
#     + SE_coords_df['exon_start'].astype("str") + "-" \
#         + SE_coords_df['exon_end'].astype("str")

# # Add gene names from GTF
# SE_anno_df = SE_coords_df.merge(gtf_gene[['gene_id', 'gene_name']], on="gene_id", how="left")
# SE_anno_df.index = SE_coords_df.index

# pickle.dump(SE_anno_df, open('data/SE_anno_df.pkl', 'wb'))

/tmp/ipykernel_181351/3782311491.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  SE_coords_df = intron_coords_df.groupby("SE").apply(safe_SE_coords)


In [ ]:
SE_anno_df = pickle.load(open('data/SEs_annotated.pkl', 'rb'))

In [ ]:
# Add annotations to cell type SEs

ct_columns = ['CGE Class', 'All GABAergic', 'All Neuronal',
              'Upper layer glutamatergic', 'Deep layer glutamatergic', 
              'Oligo', 'OPC', 'Astro', 'Micro/PVM', 'Endo']


column_order = ['gene_name', 'exon_len', 'exon_coords', 'intron_end', 
                'intron_start', 'is_specific', 'specific_direction', 'r', 'fdr'] \
                    + ct_columns

def _safe(name):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(name))

ctype_specific_SEs = {}
for ct, results in combined_results.items():
    results_anno = results.join(SE_anno_df, how="left")
    diff_columns = results.columns[results.columns.str.contains("diff")].tolist()
    results_anno_ordered = results_anno[column_order + diff_columns]
    ctype_specific_SEs[ct] = results_anno_ordered 
    results_anno_ordered.to_csv(f"data/ct_SEs/{data_source}_{_safe(ct)}_SEs.csv")

In [19]:
pickle.dump(ctype_specific_SEs, open(f"data/{data_source}_ctype_specific_SEs.pkl", "wb"))